# Interface de Data Discovery: Jupyter + SQL
Podemos transformar o nosso JupyterLab numa autêntica *Worksheet* de Data Warehouse como o **AWS Athena**.

Para isso, usamos a extensão Mágica do Python para habilitar blocos de código nativos SQL.

In [15]:
# Carrega a extensão Mágica do SQL
%load_ext sql

# Inicializa a conexão SQLAlchemy com o seu Trino (Redshift local)
%sql trino://jovyan@trino:8080/minio

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


--- 
## Suas Consultas Livres (*Ad-Hoc*)
A partir de agora, as células que começam com `%%sql` deixam de ser Python e são enviadas **diretamente ao Trino**. O resultado é renderizado em HTML idêntico ao de uma ferramenta gráfica de Client SQL:

In [16]:
%%sql
SHOW SCHEMAS

Running query in 'trino://jovyan@trino:8080/minio'

Schema
information_schema
raw
trusted


In [17]:
%%sql

SHOW COLUMNS FROM minio.trusted.penguins

Running query in 'trino://jovyan@trino:8080/minio'

Column,Type,Extra,Comment
especies,varchar,,
ilha,varchar,,
bico_comp_mm,double,,
bico_largura_mm,double,,
nadadeira_comp_mm,double,,
masso_corporal_g,double,,
massa_corporal_kg,double,,
sexo,varchar,,
ano,bigint,,


In [18]:
%%sql

SELECT *
FROM minio.trusted.penguins

Running query in 'trino://jovyan@trino:8080/minio'

especies,ilha,bico_comp_mm,bico_largura_mm,nadadeira_comp_mm,masso_corporal_g,massa_corporal_kg,sexo,ano
adelie,torgersen,39.1,18.7,181.0,3750.0,3.75,macho,2007
adelie,torgersen,39.5,17.4,186.0,3800.0,3.8,femea,2007
adelie,torgersen,40.3,18.0,195.0,3250.0,3.25,femea,2007
adelie,torgersen,36.7,19.3,193.0,3450.0,3.45,femea,2007
adelie,torgersen,39.3,20.6,190.0,3650.0,3.65,macho,2007
adelie,torgersen,38.9,17.8,181.0,3625.0,3.625,femea,2007
adelie,torgersen,39.2,19.6,195.0,4675.0,4.675,macho,2007
adelie,torgersen,41.1,17.6,182.0,3200.0,3.2,femea,2007
adelie,torgersen,38.6,21.2,191.0,3800.0,3.8,macho,2007
adelie,torgersen,34.6,21.1,198.0,4400.0,4.4,macho,2007


In [19]:
%%sql
SELECT *
FROM minio.trusted.penguins
WHERE ilha = 'torgersen'
ORDER BY nadadeira_comp_mm DESC
LIMIT 5


Running query in 'trino://jovyan@trino:8080/minio'

especies,ilha,bico_comp_mm,bico_largura_mm,nadadeira_comp_mm,masso_corporal_g,massa_corporal_kg,sexo,ano
adelie,torgersen,44.1,18.0,210.0,4000.0,4.0,macho,2009
adelie,torgersen,41.4,18.5,202.0,3875.0,3.875,macho,2009
adelie,torgersen,40.6,19.0,199.0,4000.0,4.0,macho,2009
adelie,torgersen,37.3,20.5,199.0,3775.0,3.775,macho,2009
adelie,torgersen,37.7,19.8,198.0,3500.0,3.5,macho,2009


In [20]:
%%sql

SELECT 
    'RAW (Bruto/Sujo)' AS camada,
    especies, 
    COUNT(*) AS total_pinguins 
FROM minio.raw.penguins 
GROUP BY especies

UNION ALL

SELECT 
    'TRUSTED (Curado/Limpo)' AS camada,
    especies, 
    COUNT(*) AS total_pinguins 
FROM minio.trusted.penguins 
GROUP BY especies

ORDER BY especies, camada


Running query in 'trino://jovyan@trino:8080/minio'

camada,especies,total_pinguins
RAW (Bruto/Sujo),adelie,152
TRUSTED (Curado/Limpo),adelie,146
RAW (Bruto/Sujo),chinstrap,68
TRUSTED (Curado/Limpo),chinstrap,68
RAW (Bruto/Sujo),gentoo,124
TRUSTED (Curado/Limpo),gentoo,119


In [21]:
%%sql
SELECT especies, count(*) as total 
FROM minio.raw.penguins 
GROUP BY especies 
ORDER BY total DESC


Running query in 'trino://jovyan@trino:8080/minio'

especies,total
adelie,152
gentoo,124
chinstrap,68


In [22]:
%%sql
SELECT especies, count(*) as total 
FROM minio.trusted.penguins 
GROUP BY especies 
ORDER BY total DESC


Running query in 'trino://jovyan@trino:8080/minio'

especies,total
adelie,146
gentoo,119
chinstrap,68
